### Setup & Feature Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

# 1. Load Market Data
df_clean = pd.read_parquet('group2_stocks.parquet')
df_benchmark = pd.read_parquet('benchmark.parquet')
log_returns = np.log(df_clean / df_clean.shift(1)).fillna(0)
tickers = df_clean.columns.tolist()

# --- NEW: NORMALIZATION FUNCTION ---
def load_and_normalize(filename):
    df = pd.read_csv(filename)
    
    # Standardize the Index (Ticker)
    if 'Ticker' in df.columns:
        df = df.set_index('Ticker')
    
    # Map variations (like 'Style_Cluster') to 'Cluster'
    rename_map = {
        'Style_Cluster': 'Cluster',
        'cluster': 'Cluster',
        'labels': 'Cluster'
    }
    df = df.rename(columns=rename_map)
    
    # Final Safety: If 'Cluster' isn't there, take the first available column
    if 'Cluster' not in df.columns:
        df = df.rename(columns={df.columns[0]: 'Cluster'})
        
    return df[['Cluster']] # Prune extra columns (PE, DE, etc.)

# 2. Load the 3 Clustering "DNA" Profiles using the new function
km_labels = load_and_normalize('labels_kmeans.csv')
dec_labels = load_and_normalize('labels_dec.csv')
style_dec_labels = load_and_normalize('labels_style_dec.csv')

print(f"✅ Setup Complete. All schemas normalized to 'Cluster'.")
print(f"Active Tickers: {len(tickers)}")

### The Optimization Engine (Your Logic)

In [ ]:
TRADING_DAYS = 252
risk_free_rate = 0.04 
cov_matrix = log_returns.cov() * TRADING_DAYS
expected_returns = log_returns.mean() * TRADING_DAYS

def negative_utility(weights, exp_rets, cov_mat, gamma):
    port_ret = np.dot(weights, exp_rets)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_mat, weights)))
    utility = port_ret - (gamma / 2) * (port_vol**2)
    return -utility

def get_optimal_weights(tickers_subset, gamma=2.5):
    subset_rets = expected_returns[tickers_subset]
    subset_cov = cov_matrix.loc[tickers_subset, tickers_subset]
    n_assets = len(tickers_subset)
    
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0.0, 1.0) for _ in range(n_assets))
    init_guess = [1.0 / n_assets] * n_assets
    
    result = minimize(negative_utility, init_guess, 
                      args=(subset_rets, subset_cov, gamma),
                      method='SLSQP', bounds=bounds, constraints=constraints)
    return dict(zip(tickers_subset, result.x))

### Constructing the Three Strategies

In [ ]:
def build_strategy(labels_df, name):
    print(f"Optimizing {name}...")
    # Extract tickers for Cluster 0 and 1
    c0 = labels_df[labels_df['Cluster'] == 0].index.tolist()
    c1 = labels_df[labels_df['Cluster'] == 1].index.tolist()
    
    # Solve for individual clusters
    w0 = get_optimal_weights(c0)
    w1 = get_optimal_weights(c1)
    
    # Combine with 60/40 weighting
    portfolio = {}
    for t, w in w0.items(): portfolio[t] = w * 0.60
    for t, w in w1.items(): portfolio[t] = w * 0.40
    return portfolio

# Build the 3 portfolios
port_kmeans = build_strategy(km_labels, "Traditional K-Means")
port_dec = build_strategy(dec_labels, "Deep DEC")
port_style = build_strategy(style_dec_labels, "StyleDEC (Triple)")
port_equal = {t: 1.0/len(tickers) for t in tickers} # Benchmark

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_portfolio(weights_dict, name, color_palette='viridis'):
    """
    Translates portfolio weights into risk/return metrics and visualizes the top 15 allocations.
    """
    # 1. Vectorize the weights to align with our expected_returns Series
    # We ensure that tickers not in the portfolio are treated as 0% weight
    w_vector = np.array([weights_dict.get(t, 0.0) for t in expected_returns.index])
    
    # 2. Performance Math
    # Expected Return: Dot product of weights and mean returns
    ret = np.dot(w_vector, expected_returns)
    
    # Portfolio Volatility: sqrt(W^T * Cov * W)
    vol = np.sqrt(np.dot(w_vector.T, np.dot(cov_matrix, w_vector)))
    
    # Sharpe Ratio: (Return - Risk-Free Rate) / Volatility
    sharpe = (ret - risk_free_rate) / vol
    
    actual_max_weight = np.max(w_vector)
    
    # 3. Preparation for Visualization
    holdings_df = pd.DataFrame(list(weights_dict.items()), columns=['Ticker', 'Weight'])
    active_holdings = holdings_df[holdings_df['Weight'] > 0.001].sort_values('Weight', ascending=False)
    
    print(f"\n--- {name} ANALYSIS ---")
    print(f"Total Selected Stocks (>0.1%): {len(active_holdings)}")
    
    # 4. Plotting the Strategy Structure
    if not active_holdings.empty:
        plt.figure(figsize=(12, 5))
        sns.barplot(data=active_holdings.head(15), x='Ticker', y='Weight', palette=color_palette)
        plt.title(f"{name}: Top 15 Allocations (Group 2)", fontsize=14)
        plt.ylabel("Weighting (%)")
        plt.xticks(rotation=45)
        plt.grid(axis='y', linestyle='--', alpha=0.3)
        plt.show()
    
    return {
        'Name': name,
        'Return': ret,
        'Volatility': vol,
        'Sharpe Ratio': sharpe,
        'Max Weight': actual_max_weight
    }

In [ ]:
# Re-use your 'analyze_portfolio' function here...
stats_KM = analyze_portfolio(port_kmeans, "K-Means Portfolio", 'Reds_r')
stats_DEC = analyze_portfolio(port_dec, "DEC Portfolio", 'Blues_r')
stats_Style = analyze_portfolio(port_style, "StyleDEC Portfolio", 'Purples_r')
stats_Bench = analyze_portfolio(port_equal, "Equal-Weight Benchmark", 'Greens_r')

comparison_df = pd.DataFrame([stats_KM, stats_DEC, stats_Style, stats_Bench]).set_index('Name')
display(comparison_df.round(4))